# Dam Inspection Commentary — Phase 0: Model Selection Experiment

## Objective
Scientifically compare three Vision-Language Models (VLMs) on dam inspection imagery under zero-shot conditions.  
The best-performing model will be selected for domain-specific fine-tuning in Phase 1.

## Candidates
| Model | Parameters | Provider |
|---|---|---|
| PaliGemma2 3B | 3B | Google DeepMind |
| LLaVA-Phi-3 mini | 3.8B | Microsoft / LLaVA team |
| Moondream2 | 1.9B | Vikhyat Kopf |

## Protocol
- Same prompt for all models
- Same set of proxy dam images
- Outputs saved to JSON for downstream evaluation (Phase 2)

## Runtime Requirements
- GPU: T4 (16GB VRAM) — Google Colab free tier
- Each model is loaded, evaluated, then explicitly unloaded before the next
- 4-bit quantization (bitsandbytes) used for all models to fit within VRAM budget

## 0. Environment Setup

In [ ]:
# Verify GPU availability before anything else
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — switch runtime to T4 GPU')

In [ ]:
%%capture
!pip install transformers accelerate bitsandbytes Pillow requests torch torchvision
!pip install moondream  # moondream2 has its own pip package
!pip install rouge-score bert-score nltk  # for Phase 2 metrics, install now

In [ ]:
import gc
import json
import time
import torch
import requests
import numpy as np
from PIL import Image
from io import BytesIO
from pathlib import Path
from datetime import datetime
from transformers import (
    AutoProcessor, AutoModelForVision2Seq,
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'Device: {DEVICE}')
print(f'VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if DEVICE == 'cuda' else 'CPU only')

## 1. Proxy Dataset

Public domain / CC-licensed dam images used as a proxy for the company dataset.  
These are sourced from Wikimedia Commons and the US Bureau of Reclamation (USBR) public photo library.  
All images are structurally diverse: gravity dams, arch dams, embankment dams, spillways.

> **Note:** Replace `PROXY_IMAGES` with your local dataset paths once company data arrives.  
> The `load_image()` function accepts both URLs and local file paths.

In [ ]:
# Proxy dam images — public domain sources
# Format: (url_or_path, descriptive_id, dam_type)
PROXY_IMAGES = [
    # Gravity dams
    ('https://upload.wikimedia.org/wikipedia/commons/thumb/8/8e/Hoover_Dam_from_air.jpg/1280px-Hoover_Dam_from_air.jpg',
     'hoover_dam_aerial', 'arch-gravity'),
    ('https://upload.wikimedia.org/wikipedia/commons/thumb/1/1e/Shasta_Dam_face.jpg/1280px-Shasta_Dam_face.jpg',
     'shasta_dam_face', 'gravity'),
    ('https://upload.wikimedia.org/wikipedia/commons/thumb/b/b2/Grand_Coulee_Dam_2012.jpg/1280px-Grand_Coulee_Dam_2012.jpg',
     'grand_coulee_dam', 'gravity'),
    # Arch dams
    ('https://upload.wikimedia.org/wikipedia/commons/thumb/3/38/Glen_Canyon_Dam_abt_2013.jpg/1280px-Glen_Canyon_Dam_abt_2013.jpg',
     'glen_canyon_arch', 'arch'),
    ('https://upload.wikimedia.org/wikipedia/commons/thumb/f/f6/Vajont_Dam.jpg/1280px-Vajont_Dam.jpg',
     'vajont_arch', 'arch'),
    # Embankment / earthfill
    ('https://upload.wikimedia.org/wikipedia/commons/thumb/6/6f/Oroville_Dam.jpg/1280px-Oroville_Dam.jpg',
     'oroville_embankment', 'embankment'),
    ('https://upload.wikimedia.org/wikipedia/commons/thumb/9/9c/Tarbela_Dam.jpg/1280px-Tarbela_Dam.jpg',
     'tarbela_embankment', 'embankment'),
    # Spillways / operational
    ('https://upload.wikimedia.org/wikipedia/commons/thumb/9/97/Itaipu_binacional.jpg/1280px-Itaipu_binacional.jpg',
     'itaipu_spillway', 'gravity'),
    ('https://upload.wikimedia.org/wikipedia/commons/thumb/2/2e/Three_Gorges_Dam%2C_Yangtze_River%2C_China.jpg/1280px-Three_Gorges_Dam%2C_Yangtze_River%2C_China.jpg',
     'three_gorges_gravity', 'gravity'),
    # Smaller / regional dams (more representative of inspection scenarios)
    ('https://upload.wikimedia.org/wikipedia/commons/thumb/e/e1/Altus_Dam.jpg/1280px-Altus_Dam.jpg',
     'altus_dam', 'gravity'),
]

def load_image(source: str) -> Image.Image:
    """Load image from URL or local path. Returns PIL Image in RGB."""
    if source.startswith('http'):
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(source, headers=headers, timeout=15)
        response.raise_for_status()
        return Image.open(BytesIO(response.content)).convert('RGB')
    return Image.open(source).convert('RGB')

# Pre-fetch and cache images to avoid repeated downloads
print('Loading proxy dataset...')
dataset = []
for url, img_id, dam_type in PROXY_IMAGES:
    try:
        img = load_image(url)
        dataset.append({'id': img_id, 'dam_type': dam_type, 'image': img, 'source': url})
        print(f'  ✓ {img_id} ({dam_type}) — {img.size}')
    except Exception as e:
        print(f'  ✗ {img_id} — {e}')

print(f'\nDataset ready: {len(dataset)} images')

## 2. Engineering Prompt Design

A single standardized prompt is used across all models.  
Prompt engineering follows the **role + task + format + uncertainty** pattern,  
which is standard practice when eliciting domain-specific structured outputs from VLMs.

In [ ]:
ENGINEERING_PROMPT = """You are a licensed civil engineer specializing in dam safety inspection and structural assessment.
Analyze the provided dam image and produce a structured professional inspection commentary.

Your commentary must address the following sections:

1. STRUCTURE TYPE: Identify the dam type (gravity, arch, embankment, buttress, rockfill, etc.) and construction material.
2. VISIBLE STRUCTURAL ELEMENTS: Describe observable components — crest, downstream face, abutments, spillway, stilling basin, outlets.
3. SURFACE & STRUCTURAL CONDITION: Note any visible cracking, spalling, seepage staining, erosion, settlement, or material deterioration.
4. VEGETATION & ENCROACHMENT: Assess presence of vegetation on the embankment or structure body and proximity of encroachments.
5. HYDRAULIC OBSERVATIONS: Describe reservoir level, downstream flow conditions, and any signs of overtopping risk or scour.
6. ANOMALIES & AREAS OF CONCERN: Flag any observable defects, deformations, or conditions requiring further investigation.
7. RECOMMENDED ACTIONS: Propose immediate, short-term, and long-term inspection or maintenance actions.

Use formal engineering language throughout. Be precise and quantitative where possible.
If a feature is not visible or cannot be reliably assessed from the image, explicitly state: 'Not assessable from available imagery.'
Do not speculate beyond what is visually observable."""

print('Engineering prompt defined.')
print(f'Prompt length: {len(ENGINEERING_PROMPT.split())} words')

## 3. Utility Functions

In [ ]:
def clear_gpu_memory():
    """Aggressively free VRAM between model loads."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    time.sleep(2)  # give the allocator time to settle
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'VRAM after cleanup: {used:.2f} / {total:.1f} GB used')

def save_results(results: dict, filename: str):
    """Persist results to JSON, excluding non-serializable image objects."""
    serializable = {}
    for model_name, outputs in results.items():
        serializable[model_name] = [
            {k: v for k, v in entry.items() if k != 'image'}
            for entry in outputs
        ]
    path = OUTPUT_DIR / filename
    with open(path, 'w') as f:
        json.dump(serializable, f, indent=2)
    print(f'Results saved to {path}')

def get_bnb_config() -> BitsAndBytesConfig:
    """Standard 4-bit quantization config for T4 VRAM budget."""
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

# Shared results store
all_results = {}
print('Utilities ready.')

## 4. Model 1 — PaliGemma2 3B

**Architecture:** SigLIP vision encoder (400M) + Gemma2 2B language model  
**Fine-tuning target:** This is the primary candidate. PaliGemma2 was designed for VQA and dense captioning tasks and supports instruction fine-tuning via LoRA/QLoRA natively through HuggingFace PEFT.

> Requires HuggingFace token — accept the model license at: https://huggingface.co/google/paligemma2-3b-pt-224

In [ ]:
from huggingface_hub import login

# Authenticate — required for gated models (PaliGemma2, Phi-3)
# Get your token at https://huggingface.co/settings/tokens
HF_TOKEN = ''  # ← paste your HuggingFace token here
login(token=HF_TOKEN)

In [ ]:
PALIGEMMA_MODEL_ID = 'google/paligemma2-3b-pt-224'

print('Loading PaliGemma2 3B (4-bit quantized)...')
pg_processor = AutoProcessor.from_pretrained(PALIGEMMA_MODEL_ID, token=HF_TOKEN)
pg_model = AutoModelForVision2Seq.from_pretrained(
    PALIGEMMA_MODEL_ID,
    quantization_config=get_bnb_config(),
    device_map='auto',
    token=HF_TOKEN,
)
pg_model.eval()
print('PaliGemma2 loaded.')
clear_gpu_memory()

In [ ]:
def run_paligemma(image: Image.Image, prompt: str, max_new_tokens: int = 512) -> str:
    inputs = pg_processor(
        text=prompt,
        images=image,
        return_tensors='pt'
    ).to(DEVICE)

    input_len = inputs['input_ids'].shape[-1]

    with torch.inference_mode():
        output = pg_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # greedy for reproducibility
        )

    # Decode only the newly generated tokens
    generated = output[0][input_len:]
    return pg_processor.decode(generated, skip_special_tokens=True).strip()


print('Running PaliGemma2 on proxy dataset...')
pg_results = []

for sample in dataset:
    start = time.time()
    commentary = run_paligemma(sample['image'], ENGINEERING_PROMPT)
    elapsed = time.time() - start

    pg_results.append({
        'id': sample['id'],
        'dam_type': sample['dam_type'],
        'model': 'paligemma2-3b',
        'commentary': commentary,
        'inference_time_s': round(elapsed, 2),
        'word_count': len(commentary.split()),
    })
    print(f"  [{sample['id']}] {elapsed:.1f}s — {len(commentary.split())} words")

all_results['paligemma2-3b'] = pg_results
save_results({'paligemma2-3b': pg_results}, 'paligemma2_results.json')

# Print a sample output
print('\n--- Sample Output (first image) ---')
print(pg_results[0]['commentary'])

In [ ]:
# Unload PaliGemma2 before loading next model
del pg_model, pg_processor
clear_gpu_memory()

## 5. Model 2 — LLaVA-Phi-3 Mini (3.8B)

**Architecture:** CLIP ViT-L vision encoder + Phi-3 Mini 3.8B language model  
**Rationale for inclusion:** Phi-3 Mini has strong instruction-following and reasoning capabilities relative to its size, making it a competitive candidate for structured commentary generation.

In [ ]:
LLAVA_PHI3_MODEL_ID = 'microsoft/llava-phi-3-mini'

print('Loading LLaVA-Phi-3 Mini (4-bit quantized)...')
lp_processor = AutoProcessor.from_pretrained(LLAVA_PHI3_MODEL_ID, token=HF_TOKEN)
lp_model = AutoModelForCausalLM.from_pretrained(
    LLAVA_PHI3_MODEL_ID,
    quantization_config=get_bnb_config(),
    device_map='auto',
    token=HF_TOKEN,
    trust_remote_code=True,  # required for Phi-3
)
lp_model.eval()
print('LLaVA-Phi-3 Mini loaded.')
clear_gpu_memory()

In [ ]:
def run_llava_phi3(image: Image.Image, prompt: str, max_new_tokens: int = 512) -> str:
    # LLaVA-Phi-3 uses a chat template with <image> token
    messages = [
        {'role': 'user', 'content': f'<image>\n{prompt}'}
    ]
    text = lp_processor.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = lp_processor(text=text, images=image, return_tensors='pt').to(DEVICE)
    input_len = inputs['input_ids'].shape[-1]

    with torch.inference_mode():
        output = lp_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated = output[0][input_len:]
    return lp_processor.decode(generated, skip_special_tokens=True).strip()


print('Running LLaVA-Phi-3 Mini on proxy dataset...')
lp_results = []

for sample in dataset:
    start = time.time()
    commentary = run_llava_phi3(sample['image'], ENGINEERING_PROMPT)
    elapsed = time.time() - start

    lp_results.append({
        'id': sample['id'],
        'dam_type': sample['dam_type'],
        'model': 'llava-phi3-mini',
        'commentary': commentary,
        'inference_time_s': round(elapsed, 2),
        'word_count': len(commentary.split()),
    })
    print(f"  [{sample['id']}] {elapsed:.1f}s — {len(commentary.split())} words")

all_results['llava-phi3-mini'] = lp_results
save_results({'llava-phi3-mini': lp_results}, 'llava_phi3_results.json')

print('\n--- Sample Output (first image) ---')
print(lp_results[0]['commentary'])

In [ ]:
del lp_model, lp_processor
clear_gpu_memory()

## 6. Model 3 — Moondream2 (1.9B)

**Architecture:** SigLIP vision encoder + Phi-1.5 language model  
**Rationale for inclusion:** Establishes the performance floor for lightweight VLMs. If Moondream2 performs comparably to larger models, it would be preferred for deployment due to its inference efficiency.

In [ ]:
import moondream as md

print('Loading Moondream2...')
# Moondream2 has its own API — does not use HuggingFace generate() directly
moon_model = md.vl(model='moondream-2b-int8.mf')  # 4-bit quantized variant
print('Moondream2 loaded.')

In [ ]:
def run_moondream(image: Image.Image, prompt: str) -> str:
    # Moondream encodes image first, then queries it
    encoded = moon_model.encode_image(image)
    result = moon_model.query(encoded, prompt)
    return result.get('answer', '').strip()


print('Running Moondream2 on proxy dataset...')
md_results = []

for sample in dataset:
    start = time.time()
    commentary = run_moondream(sample['image'], ENGINEERING_PROMPT)
    elapsed = time.time() - start

    md_results.append({
        'id': sample['id'],
        'dam_type': sample['dam_type'],
        'model': 'moondream2',
        'commentary': commentary,
        'inference_time_s': round(elapsed, 2),
        'word_count': len(commentary.split()),
    })
    print(f"  [{sample['id']}] {elapsed:.1f}s — {len(commentary.split())} words")

all_results['moondream2'] = md_results
save_results({'moondream2': md_results}, 'moondream2_results.json')

print('\n--- Sample Output (first image) ---')
print(md_results[0]['commentary'])

In [ ]:
del moon_model
clear_gpu_memory()

## 7. Preliminary Quantitative Comparison

Automatic metrics computed here are **proxies only** — they measure output characteristics,  
not engineering correctness. The full evaluation rubric is applied in Phase 2.

Metrics computed:
- **Avg. word count** — verbosity proxy
- **Section coverage** — % of the 7 required sections present in output
- **Technical term density** — frequency of domain-specific vocabulary
- **Avg. inference time** — deployment feasibility

In [ ]:
# Load all saved results (works even if models were run in separate sessions)
RESULT_FILES = {
    'paligemma2-3b': 'paligemma2_results.json',
    'llava-phi3-mini': 'llava_phi3_results.json',
    'moondream2': 'moondream2_results.json',
}

loaded_results = {}
for model_name, fname in RESULT_FILES.items():
    fpath = OUTPUT_DIR / fname
    if fpath.exists():
        with open(fpath) as f:
            data = json.load(f)
        loaded_results[model_name] = data.get(model_name, [])
    else:
        print(f'Warning: {fname} not found — run the corresponding model cell first.')

print(f'Loaded results for: {list(loaded_results.keys())}')

In [ ]:
# Engineering domain vocabulary for technical density scoring
ENGINEERING_TERMS = {
    # Structure types
    'gravity', 'arch', 'embankment', 'buttress', 'rockfill', 'earthfill', 'cofferdam',
    # Structural components
    'crest', 'abutment', 'spillway', 'stilling', 'basin', 'outlet', 'parapet',
    'downstream', 'upstream', 'foundation', 'grouting', 'drainage', 'weir',
    # Defects
    'cracking', 'spalling', 'seepage', 'erosion', 'settlement', 'subsidence',
    'scour', 'piping', 'liquefaction', 'overtopping', 'deformation', 'deterioration',
    # Inspection terminology
    'assessment', 'inspection', 'monitoring', 'remediation', 'instrumentation',
    'piezometer', 'inclinometer', 'geodetic', 'structural', 'geotechnical',
    # Actions
    'investigate', 'remediate', 'reinforce', 'rehabilitate', 'dewater',
}

REQUIRED_SECTIONS = [
    'structure type',
    'structural element',
    'condition',
    'vegetation',
    'hydraulic',
    'concern',
    'recommend',
]

def score_section_coverage(text: str) -> float:
    """Fraction of required sections present (keyword match)."""
    text_lower = text.lower()
    found = sum(1 for kw in REQUIRED_SECTIONS if kw in text_lower)
    return round(found / len(REQUIRED_SECTIONS), 3)

def score_technical_density(text: str) -> float:
    """Engineering term occurrences per 100 words."""
    words = text.lower().split()
    if not words:
        return 0.0
    hits = sum(1 for w in words if w.strip('.,;:()') in ENGINEERING_TERMS)
    return round((hits / len(words)) * 100, 2)


# Compute summary statistics per model
summary = {}
for model_name, outputs in loaded_results.items():
    if not outputs:
        continue
    word_counts = [o['word_count'] for o in outputs]
    inf_times = [o['inference_time_s'] for o in outputs]
    section_scores = [score_section_coverage(o['commentary']) for o in outputs]
    tech_scores = [score_technical_density(o['commentary']) for o in outputs]

    summary[model_name] = {
        'avg_word_count': round(np.mean(word_counts), 1),
        'avg_inference_time_s': round(np.mean(inf_times), 2),
        'avg_section_coverage': round(np.mean(section_scores), 3),
        'avg_technical_density': round(np.mean(tech_scores), 2),
        'n_samples': len(outputs),
    }

print('\n=== PHASE 0 — Preliminary Comparison ===')
print(f'{"Metric":<30} ' + '  '.join(f'{k:<20}' for k in summary.keys()))
print('-' * 90)
metrics = ['avg_word_count', 'avg_inference_time_s', 'avg_section_coverage', 'avg_technical_density', 'n_samples']
for metric in metrics:
    row = f'{metric:<30} '
    for model_name in summary:
        val = summary[model_name].get(metric, 'N/A')
        row += f'{str(val):<22}'
    print(row)

# Save summary
with open(OUTPUT_DIR / 'phase0_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('\nSummary saved to outputs/phase0_summary.json')

## 8. Side-by-Side Output Inspection

Visual inspection of outputs for the same image across all three models.  
This is the qualitative complement to the automatic metrics above.

In [ ]:
def compare_outputs(image_id: str):
    """Print all model outputs for a given image side by side."""
    print(f'\n{'='*80}')
    print(f'IMAGE: {image_id}')
    print(f'{'='*80}')
    for model_name, outputs in loaded_results.items():
        entry = next((o for o in outputs if o['id'] == image_id), None)
        if entry:
            print(f'\n--- {model_name.upper()} ({entry["word_count"]} words, {entry["inference_time_s"]}s) ---')
            print(entry['commentary'])
        else:
            print(f'\n--- {model_name.upper()} --- (no output found)')

# Inspect outputs for first image
if dataset:
    compare_outputs(dataset[0]['id'])

# To compare all images:
# for sample in dataset:
#     compare_outputs(sample['id'])

## 9. Model Selection Decision

Based on the quantitative comparison and qualitative inspection above, complete this cell  
with your selection and justification. This becomes Section X of your project report.

In [ ]:
# ── Fill this in after reviewing results ──────────────────────────────────────
SELECTED_MODEL = 'paligemma2-3b'  # UPDATE after evaluation

SELECTION_RATIONALE = """
[To be completed after Phase 0 evaluation]

Example structure:
PaliGemma2-3B was selected for fine-tuning based on the following criteria:
1. Highest section coverage score (X/7 sections present on average)
2. Highest technical term density (X per 100 words)
3. Acceptable inference time (Xs avg on T4)
4. Architecture suitability: SigLIP encoder provides stronger visual grounding
   than CLIP for structural/texture analysis relevant to dam inspection.
5. Fine-tuning ecosystem: Native HuggingFace PEFT support with documented
   QLoRA fine-tuning pipeline.
"""

decision = {
    'selected_model': SELECTED_MODEL,
    'rationale': SELECTION_RATIONALE,
    'timestamp': datetime.now().isoformat(),
    'phase0_summary': summary,
}

with open(OUTPUT_DIR / 'model_selection_decision.json', 'w') as f:
    json.dump(decision, f, indent=2)

print(f'Selected model: {SELECTED_MODEL}')
print(f'Decision saved to outputs/model_selection_decision.json')
print('\n→ Proceed to Phase 1: Fine-tuning Pipeline')

---
## Phase 0 Complete

**Outputs produced:**
- `outputs/paligemma2_results.json` — PaliGemma2 inference outputs
- `outputs/llava_phi3_results.json` — LLaVA-Phi-3 Mini inference outputs  
- `outputs/moondream2_results.json` — Moondream2 inference outputs
- `outputs/phase0_summary.json` — Comparative metrics summary
- `outputs/model_selection_decision.json` — Documented model selection

**Next:** Phase 1 — QLoRA Fine-tuning Pipeline on selected model with company dataset.